# AG983 | Class 1 Workshop
## Text Pre-Processing: Decisions and Consequences

**University of Strathclyde Business School** | Dr James Bowden

---

This workshop accompanies Section 6.4 of the Class 1 lecture. You will work through a text pre-processing pipeline, making decisions at each step and observing their consequences for subsequent analysis.

The corpus is drawn from the Enron email dataset (Klimt & Yang, 2004), released into the public domain by the Federal Energy Regulatory Commission during the 2001 Enron investigation. It was selected because it combines the hedged, forward-looking register typical of corporate financial disclosure with the informal directness of internal correspondence. That combination exposes the limitations of generic preprocessing pipelines in ways that a standard newswire corpus would not.

As Renault (2020) demonstrates, optimal preprocessing increases the sentiment-returns correlation by approximately 55% relative to a naive baseline. Pipeline quality explains more variance in downstream performance than algorithm choice.

**Instructions:** Run each cell in order using Shift+Enter. Make your choices where indicated and record your observations in the accompanying worksheet.

**Note:** Run cells strictly from top to bottom. Variables set in earlier steps are used in later ones.

---
## Step 0: Environment Setup

Run this cell first. It installs the required packages and loads the corpus and both sentiment dictionaries.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install",
                "nltk", "pandas", "matplotlib", "seaborn", "wordcloud", "-q"],
               check=True)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re, warnings
from collections import Counter
warnings.filterwarnings("ignore")

import nltk
for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4",
            "averaged_perceptron_tagger"]:
    try:    nltk.download(pkg, quiet=True)
    except: pass

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords as nltk_stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

# ── Load corpus ────────────────────────────────────────────────────────────
CORPUS_URL = ("https://raw.githubusercontent.com/iamjamesbowden/AG983/"
              "main/class1/data/enron_sample.csv")
corpus = pd.read_csv(CORPUS_URL)

# ── Load dictionaries ──────────────────────────────────────────────────────
LM_URL = ("https://raw.githubusercontent.com/iamjamesbowden/AG952/"
          "main/assignments/march2026/data/lm_dictionary.csv")
HV_URL = ("https://raw.githubusercontent.com/iamjamesbowden/AG952/"
          "main/assignments/march2026/data/harvard_iv_dictionary.csv")
lm  = pd.read_csv(LM_URL)
hiv = pd.read_csv(HV_URL)

LM_NEGATIVE = set(lm[lm["Negative"] == 1]["Word"].str.lower())
LM_POSITIVE = set(lm[lm["Positive"] == 1]["Word"].str.lower())
HV_NEGATIVE = set(hiv[hiv["Negative"] == 1]["Word"].str.lower())
HV_POSITIVE = set(hiv[hiv["Positive"] == 1]["Word"].str.lower())

# ── Stop-word lists ────────────────────────────────────────────────────────
STD_SW = set(nltk_stopwords.words("english"))

# Finance-adjusted list: explicitly retain ALL terms that carry analytical
# weight in financial text, regardless of whether any given NLTK version
# removes them. This makes the pipeline version-stable and transparent.
# Sources: Loughran & McDonald (2011), Renault (2020).
FINANCE_PROTECTED = {
    # Modal verbs - carry forward-looking, uncertainty, and commitment signals
    "will", "would", "may", "might", "could", "can",
    "shall", "should", "must",
    # Negation - reverses meaning: "not material", "no guarantee", "nor"
    "not", "no", "nor", "never", "neither",
    # Directional terms - specific meaning in investment contexts
    "up", "down", "above", "below", "over", "under",
    # Other finance-significant function words
    "more", "less", "very", "against", "before", "after",
    "during", "between", "through", "further", "however",
    "although", "despite", "unless", "until",
}
FIN_SW = STD_SW - FINANCE_PROTECTED

# ── Helpers ────────────────────────────────────────────────────────────────
stemmer    = PorterStemmer()
lemmatizer = WordNetLemmatizer()

plt.rcParams.update({
    "figure.facecolor": "#0f0f1a", "axes.facecolor": "#0f0f1a",
    "axes.edgecolor":   "#444",    "axes.labelcolor": "#ddd",
    "xtick.color":      "#aaa",    "ytick.color":     "#aaa",
    "text.color":       "#ddd",    "grid.color":      "#2a2a3a",
    "grid.linestyle":   "--",      "grid.alpha":      0.4,
})

# Summary of what each list removes
in_std_not_fin = sorted(FINANCE_PROTECTED & STD_SW)
print(f"Corpus loaded:              {len(corpus):>5} emails")
print(f"  Pre-crisis (2000):        {(corpus.period=='pre_crisis').sum():>5}")
print(f"  Crisis (2001):            {(corpus.period=='crisis').sum():>5}")
print(f"  Collapse (late 2001-2):   {(corpus.period=='collapse').sum():>5}")
print(f"\nLM dictionary:")
print(f"  Negative words:           {len(LM_NEGATIVE):>5}")
print(f"  Positive words:           {len(LM_POSITIVE):>5}")
print(f"Harvard IV dictionary:")
print(f"  Negative words:           {len(HV_NEGATIVE):>5}")
print(f"  Positive words:           {len(HV_POSITIVE):>5}")
print(f"\nStop-word lists:")
print(f"  Standard NLTK:            {len(STD_SW):>5} words")
print(f"  Finance-adjusted:         {len(FIN_SW):>5} words")
print(f"  Finance terms in NLTK list: {in_std_not_fin}")
print("\nEnvironment ready.")

---
## Step 1: Explore the Corpus

Before processing any text, it is worth understanding what you are working with. Run the cells below to examine the corpus structure, email length distribution, and a sample of raw text.

*Record your observations for Worksheet Question 1.*

In [ ]:
print("=" * 55)
print("CORPUS OVERVIEW")
print("=" * 55)
print(corpus.groupby(["period","category"]).size().rename("emails").to_string())
print(f"\nTotal emails:  {len(corpus)}")
print(f"Total words:   {corpus.word_count.sum():,}")
print(f"Mean length:   {corpus.word_count.mean():.0f} words")
print(f"Shortest:      {corpus.word_count.min()} words")
print(f"Longest:       {corpus.word_count.max()} words")

COLORS = {"pre_crisis": "#3b82f6", "crisis": "#f59e0b", "collapse": "#ef4444"}
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.patch.set_facecolor("#0f0f1a")
for ax, (groupcol, title) in zip(axes, [("period","Email length by time period"),
                                          ("category","Email length by category")]):
    for name, grp in corpus.groupby(groupcol):
        ax.hist(grp["word_count"], bins=12, alpha=0.6,
                label=name, color=COLORS.get(name, "#8b5cf6"))
    ax.set_title(title, color="#ddd", pad=8)
    ax.set_xlabel("Word count")
    ax.set_ylabel("Frequency")
    ax.legend(fontsize=7)
plt.suptitle("Corpus structure - Enron email sample (Klimt & Yang, 2004)",
             color="#ddd", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Show a raw email - no processing applied whatsoever
# Try to pick a substantive email (word_count > 60)
sample = corpus[corpus.word_count > 60].iloc[2]
print(f"From:     {sample['sender']}")
print(f"Date:     {sample['date']}")
print(f"Period:   {sample['period']}")
print(f"Category: {sample['category']}")
print(f"Subject:  {sample['subject']}")
print(f"Words:    {sample['word_count']}")
print("-" * 60)
print(sample["body"])
print()
print("Consider this email before any processing is applied.")
print("What would need to be resolved before feeding this text to a model?")

---
## Step 2: Tokenisation

Tokenisation converts raw text into discrete units that a model can count and compare. Two common approaches are available here:

- **Whitespace split:** fast and requires no additional library, but it treats `"liability."` and `"liability"` as separate tokens because the full stop remains attached.
- **NLTK word_tokenize:** separates punctuation from words correctly and handles contractions, producing cleaner tokens for dictionary matching.

Run the cell below and compare the outputs on the same email.

In [ ]:
# Use the same email as Step 1
text = sample["body"]

ws_tokens = text.split()
nl_tokens  = word_tokenize(text.lower())

print("RAW TEXT (first 300 chars):")
print(text[:300])
print()
print(f"Whitespace split:    {len(ws_tokens):>4} tokens")
print(f"NLTK word_tokenize: {len(nl_tokens):>4} tokens")
print()

# What tokens differ? Focus on punctuation attachment
ws_set = set(t.lower() for t in ws_tokens)
nl_set = set(nl_tokens)
in_ws_not_nl = sorted([t for t in ws_set - nl_set if len(t) > 2])
print("Tokens in whitespace split but absent from NLTK output (punctuation attached):")
print("  ", in_ws_not_nl[:15])
print()
print("A whitespace tokeniser produces 'liability.' as a single token.")
print("That form will not match 'liability' in the LM dictionary.")
print("NLTK separates the punctuation, so 'liability' matches correctly.")

*Worksheet Q2: Why does the tokenisation method matter before applying a sentiment dictionary? What would happen to the token `"losses."` under whitespace splitting when matched against the Loughran-McDonald lexicon?*

---
## Step 3: Case Folding and Punctuation Removal

**Case folding** converts all text to lowercase so that `"Revenue"` and `"revenue"` are treated as a single token type.

**Punctuation removal** prevents the same word appearing as distinct tokens depending on what punctuation surrounds it.

The four panels below use the same 20 emails, so differences between panels reflect only the preprocessing step applied.

In [ ]:
def get_tokens(texts, lowercase=True, remove_punct=True):
    """Tokenise a list of texts with configurable options."""
    all_tokens = []
    for text in texts:
        if lowercase:
            text = text.lower()
        tokens = word_tokenize(text)
        if remove_punct:
            tokens = [t for t in tokens if t.isalpha()]
        all_tokens.extend(tokens)
    return all_tokens

# Use the same 20 emails for all four panels
SAMPLE_TEXTS = corpus.sample(20, random_state=983)["body"].tolist()

configs = {
    "Raw (whitespace split)":         [t for txt in SAMPLE_TEXTS for t in txt.split()],
    "NLTK tokenise only":             get_tokens(SAMPLE_TEXTS, lowercase=False, remove_punct=False),
    "Case-folded":                    get_tokens(SAMPLE_TEXTS, lowercase=True,  remove_punct=False),
    "Case-fold + punct removed":      get_tokens(SAMPLE_TEXTS, lowercase=True,  remove_punct=True),
}

fig, axes = plt.subplots(1, 4, figsize=(17, 5))
fig.patch.set_facecolor("#0f0f1a")
for ax, (label, tokens) in zip(axes, configs.items()):
    freq = Counter(tokens).most_common(15)
    words, counts = zip(*freq)
    ax.barh(list(reversed(words)), list(reversed(counts)), color="#3b82f6", alpha=0.85)
    ax.set_title(label, color="#ddd", fontsize=8.5, pad=6)
    ax.set_xlabel("Count")
plt.suptitle("Effect of case folding and punctuation removal - 20 Enron emails",
             color="#ddd", y=1.01, fontsize=10)
plt.tight_layout()
plt.show()

print("Raw whitespace tokens in the top 15 include punctuation-attached forms.")
print("After case folding and punctuation removal, all top tokens are clean alphabetic words.")
print()
print("Renault (2020) finds that retaining '!' and '?' improves financial sentiment")
print("classification by 0.3%. Not all punctuation is noise.")

---
## Step 4: Stop-Word Removal

Stop-words are high-frequency function words present in almost every document: articles, prepositions, conjunctions. Because they appear so uniformly, they carry little discriminatory power and are typically removed to reduce vocabulary size.

In financial text, however, some of these function words are analytically important. The NLTK English stop-word list removes 179 common words, including several that carry specific meaning in corporate disclosure:

| Term | Analytical role | Removed by NLTK? |
|---|---|---|
| `will` | Forward-looking commitment signal | Yes |
| `should` | Obligation or recommendation | Yes |
| `can` | Capability or possibility | Yes |
| `not` | Negation: "not material" reverses sentiment | Yes |
| `no` | Negation: "no guarantee" | Yes |
| `down` | Directional: "revenues down 8%" | Yes |
| `up` | Directional: "earnings up sharply" | Yes |

The finance-adjusted list is constructed by explicitly retaining all terms of this kind, regardless of NLTK version. Renault (2020) finds that applying the standard NLTK list reduces financial sentiment classification accuracy by 1.6%.

In [ ]:
texts = corpus["body"].tolist()
all_tokens  = get_tokens(texts, lowercase=True, remove_punct=True)
std_tokens  = [t for t in all_tokens if t not in STD_SW  and len(t) > 1]
fin_tokens  = [t for t in all_tokens if t not in FIN_SW  and len(t) > 1]

fig, axes = plt.subplots(1, 3, figsize=(17, 6))
fig.patch.set_facecolor("#0f0f1a")
for ax, (label, tokens, color) in zip(axes, [
    ("No stop-word removal",        all_tokens, "#64748b"),
    ("Standard NLTK stop-words",    std_tokens, "#ef4444"),
    ("Finance-adjusted stop-words", fin_tokens, "#22c55e"),
]):
    freq = Counter(tokens).most_common(20)
    words, counts = zip(*freq)
    ax.barh(list(reversed(words)), list(reversed(counts)), color=color, alpha=0.85)
    ax.set_title(label, color="#ddd", fontsize=9, pad=6)
    ax.set_xlabel("Count")
plt.suptitle("Stop-word removal: three approaches - full Enron corpus", color="#ddd", y=1.01)
plt.tight_layout()
plt.show()

# Show what is lost with standard stop-words
print("Terms retained by the finance-adjusted list but removed by standard NLTK:")
std_set = set(std_tokens)
fin_set = set(fin_tokens)
protected_present = sorted([t for t in FINANCE_PROTECTED if t in fin_set and t not in std_set])
print("  ", protected_present)
print()
print(f"Standard NLTK:     {len(set(std_tokens)):,} unique tokens")
print(f"Finance-adjusted:  {len(set(fin_tokens)):,} unique tokens")

In [ ]:
# ═══════════════════════════════════════════════════════
# YOUR CHOICE - change this value and re-run this cell
# Options: "none" | "standard" | "finance"
STOPWORD_CHOICE = "finance"
# ═══════════════════════════════════════════════════════

sw_map   = {"none": set(), "standard": STD_SW, "finance": FIN_SW}
ACTIVE_SW = sw_map[STOPWORD_CHOICE]
print(f"Stop-word list: '{STOPWORD_CHOICE}' ({len(ACTIVE_SW)} words removed)")
print(f"This setting carries forward to Steps 6, 7, 8, and 9.")

---
## Step 5: Number Handling

Numerical tokens introduce variance that may not reflect the linguistic signal of interest. Two sentences, "Revenue was $4.2bn" and "Revenue was $9.6bn", convey the same epistemic tone; the magnitude is captured in your quantitative data, not in your text model.

That said, not all numbers are arbitrary. Regulatory references such as "Rule 10b-5", percentage changes such as "grew 8%", and fiscal year references such as "2001" may carry meaning worth retaining. Choose your approach and observe the effect on vocabulary size.

In [ ]:
def handle_numbers(tokens, method):
    if method == "remove":
        return [t for t in tokens if not re.search(r'\d', t)]
    elif method == "replace":
        return ["NUM" if re.search(r'\d', t) else t for t in tokens]
    else:
        return tokens

# ═══════════════════════════════════════════════════════
# YOUR CHOICE - change this value and re-run this cell
# Options: "remove" | "retain" | "replace"
NUMBER_CHOICE = "remove"
# ═══════════════════════════════════════════════════════

# Apply current pipeline settings and show numeric token examples
all_tok    = get_tokens(corpus["body"].tolist(), lowercase=True, remove_punct=False)
sw_filtered = [t for t in all_tok if t not in ACTIVE_SW]
nums_found = [t for t in sw_filtered if re.search(r'\d', t)]
handled    = handle_numbers(sw_filtered, NUMBER_CHOICE)

print(f"Number handling: '{NUMBER_CHOICE}'")
print(f"Numeric tokens in corpus (sample): {sorted(set(nums_found))[:25]}")
print(f"Vocabulary before: {len(set(sw_filtered)):,}  |  After: {len(set(handled)):,}")
print(f"This setting carries forward to Steps 7, 8, and 9.")

---
## Step 6: Stemming vs Lemmatisation

Both methods reduce words to a common form, but they differ in a way that matters considerably for dictionary-based sentiment analysis.

Porter stemming (1980) removes suffixes mechanically and produces non-words. Two examples:

- `liabilities` becomes `liabil`, which is not present in the LM dictionary (which holds `liability`)
- `declining` becomes `declin`, which is not present in the LM dictionary (which holds `decline`)

Lemmatisation returns the canonical dictionary form:

- `liabilities` becomes `liability`, matching the LM entry
- `declining` becomes `decline`, matching the LM entry

The table below uses a set of financial terms drawn from the LM dictionary to illustrate where the difference is consequential.

In [ ]:
# Use key financial terms that appear in the corpus and LM dictionary
# to demonstrate the stemming-vs-lemmatisation difference clearly
FINANCIAL_TERMS = [
    # Words where stem ≠ lemma AND lemma matches LM but stem does not
    "decline",    "declining",  "adverse",    "adversely",
    "failure",    "losses",     "loss",       "concern",
    "uncertain",  "default",    "fail",       "impair",
    "deteriorate","restructure","undermine",  "achieve",
    # Words where both match (for completeness)
    "risk",       "debt",       "profit",     "growth",
]

rows = []
for token in FINANCIAL_TERMS:
    stem  = stemmer.stem(token)
    lemma = lemmatizer.lemmatize(token)
    lm_raw   = token in LM_NEGATIVE or token in LM_POSITIVE
    lm_stem  = stem  in LM_NEGATIVE or stem  in LM_POSITIVE
    lm_lemma = lemma in LM_NEGATIVE or lemma in LM_POSITIVE
    sentiment = "NEG" if (token in LM_NEGATIVE) else ("POS" if token in LM_POSITIVE else "-")
    rows.append({
        "token":      token,
        "stem":       stem,
        "lemma":      lemma,
        "LM(raw)":    "Y" if lm_raw   else "N",
        "LM(stem)":   "Y" if lm_stem  else "N",
        "LM(lemma)":  "Y" if lm_lemma else "N",
        "sentiment":  sentiment,
    })

df_norm = pd.DataFrame(rows)
print("Token normalisation and LM dictionary matching")
print("(Y = found in LM dictionary, N = not found)")
print()
print(df_norm.to_string(index=False))

stem_matches  = (df_norm["LM(stem)"]  == "Y").sum()
lemma_matches = (df_norm["LM(lemma)"] == "Y").sum()
raw_matches   = (df_norm["LM(raw)"]   == "Y").sum()
print(f"\nSummary - LM matches out of {len(FINANCIAL_TERMS)} terms:")
print(f"  Raw tokens:      {raw_matches}/{len(FINANCIAL_TERMS)}")
print(f"  After stemming: {stem_matches}/{len(FINANCIAL_TERMS)}")
print(f"  After lemma:    {lemma_matches}/{len(FINANCIAL_TERMS)}")

In [ ]:
# ═══════════════════════════════════════════════════════
# YOUR CHOICE - change this value and re-run this cell
# Options: "none" | "stemming" | "lemmatisation"
NORMALISATION_CHOICE = "lemmatisation"
# ═══════════════════════════════════════════════════════

def normalise(tokens, method):
    if method == "stemming":
        return [stemmer.stem(t) for t in tokens]
    elif method == "lemmatisation":
        return [lemmatizer.lemmatize(t) for t in tokens]
    else:
        return tokens

print(f"Normalisation: '{NORMALISATION_CHOICE}'")
print(f"This setting carries forward to Steps 7, 8, and 9.")

---
## Step 7: The Diagnostic

Your pipeline choices can now be evaluated against a concrete criterion: how many tokens in the corpus match entries in the Loughran-McDonald dictionary.

The same corpus is also run through a deliberately misconfigured pipeline, using Porter stemming with standard NLTK stop-word removal, to illustrate how many matches are lost through poor preprocessing decisions. The difference quantifies the argument made by Renault (2020) and Loughran and McDonald (2011): pipeline quality determines whether a sentiment score captures anything substantive.

In [ ]:
def run_pipeline(texts, sw_set, number_method, norm_method):
    """Apply a complete preprocessing pipeline and return token list."""
    all_tokens = []
    for text in texts:
        tokens = word_tokenize(text.lower())
        tokens = [t for t in tokens if t.isalpha() or re.search(r'\d', t)]
        tokens = handle_numbers(tokens, number_method)
        tokens = [t for t in tokens if t not in sw_set and len(t) > 1]
        tokens = normalise(tokens, norm_method)
        all_tokens.extend(tokens)
    return all_tokens

def count_lm_matches(tokens):
    pos = sum(1 for t in tokens if t in LM_POSITIVE)
    neg = sum(1 for t in tokens if t in LM_NEGATIVE)
    return pos, neg

texts = corpus["body"].tolist()

# YOUR pipeline (using your current choices)
your_tokens = run_pipeline(texts, ACTIVE_SW, NUMBER_CHOICE, NORMALISATION_CHOICE)

# Misconfigured pipeline - lecture diagnostic
bad_tokens  = run_pipeline(texts, STD_SW, "retain", "stemming")

your_pos, your_neg = count_lm_matches(your_tokens)
bad_pos,  bad_neg  = count_lm_matches(bad_tokens)

print("=" * 62)
print("PIPELINE DIAGNOSTIC - LM DICTIONARY MATCHING")
print("=" * 62)
print(f"{'':34s} {'Your':>12} {'Misconfigured':>13}")
print("-" * 62)
print(f"{'Vocabulary (unique tokens)':34s} {len(set(your_tokens)):>12,} {len(set(bad_tokens)):>13,}")
print(f"{'Total tokens':34s} {len(your_tokens):>12,} {len(bad_tokens):>13,}")
print(f"{'LM Positive matches':34s} {your_pos:>12,} {bad_pos:>13,}")
print(f"{'LM Negative matches':34s} {your_neg:>12,} {bad_neg:>13,}")
print(f"{'Total LM matches':34s} {your_pos+your_neg:>12,} {bad_pos+bad_neg:>13,}")
print("=" * 62)

improvement = (your_pos+your_neg) - (bad_pos+bad_neg)
pct = improvement / max(bad_pos+bad_neg, 1) * 100
direction = "more" if improvement > 0 else "fewer"
print(f"\nYour pipeline produces {abs(pct):.0f}% {direction} LM matches than the misconfigured version.")
print(f"\nYour settings: stop-words={STOPWORD_CHOICE}, numbers={NUMBER_CHOICE}, normalisation={NORMALISATION_CHOICE}")

In [ ]:
# Sentiment across time periods - does the signal track Enron's history?
period_results = []
for period, grp in corpus.groupby("period"):
    tokens = run_pipeline(grp["body"].tolist(), ACTIVE_SW, NUMBER_CHOICE, NORMALISATION_CHOICE)
    pos, neg = count_lm_matches(tokens)
    total = len(tokens)
    period_results.append({
        "period":       period,
        "pos_rate":     pos / total * 1000 if total > 0 else 0,
        "neg_rate":     neg / total * 1000 if total > 0 else 0,
        "net_sentiment":  (pos - neg) / total * 100 if total > 0 else 0,
    })

df_res = pd.DataFrame(period_results)
order  = ["pre_crisis", "crisis", "collapse"]
labels = ["Pre-crisis\n(2000)", "Crisis\n(2001)", "Collapse\n(late 2001-2)"]
df_res = df_res.set_index("period").reindex(order)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor("#0f0f1a")

# Panel 1: Positive and negative rates
x = np.arange(len(order))
w = 0.35
ax1.bar(x - w/2, df_res["pos_rate"], w, color="#22c55e", alpha=0.85, label="LM Positive")
ax1.bar(x + w/2, df_res["neg_rate"], w, color="#ef4444", alpha=0.85, label="LM Negative")
ax1.set_xticks(x); ax1.set_xticklabels(labels)
ax1.set_ylabel("Matches per 1,000 tokens")
ax1.set_title("LM positive and negative rates by period", color="#ddd", pad=8)
ax1.legend()

# Panel 2: Net sentiment
net = df_res["net_sentiment"].values
colors = ["#22c55e" if v >= 0 else "#ef4444" for v in net]
bars = ax2.bar(labels, net, color=colors, alpha=0.85, width=0.5)
ax2.axhline(0, color="#666", linewidth=0.8)
ax2.set_ylabel("Net LM sentiment (% of tokens)")
ax2.set_title("Net LM sentiment by period", color="#ddd", pad=8)
for bar, val in zip(bars, net):
    ax2.text(bar.get_x() + bar.get_width()/2,
             val + (0.005 if val >= 0 else -0.005),
             f"{val:+.2f}%", ha="center",
             va="bottom" if val >= 0 else "top",
             color="#ddd", fontsize=9)

plt.suptitle("LM sentiment signal by period - does it reflect Enron's financial trajectory?",
             color="#ddd", y=1.02, fontsize=10)
plt.tight_layout()
plt.show()

print("Consider whether the sentiment signal moves in the direction you would expect.")
print("These emails were written by executives and managers with clear strategic incentives.")
print("Does the LM signal reflect the financial position of the firm, or its presentation?")

---
## Step 8: Pruning

Pruning removes tokens from the document-term matrix that appear either too rarely or too frequently across the corpus, and is applied after all other cleaning steps.

- **Minimum document frequency (min_df):** tokens appearing in fewer than N documents are removed. These are typically proper nouns, idiosyncratic abbreviations, or errors with limited discriminatory value.
- **Maximum document frequency (max_df):** tokens appearing in more than a given proportion of documents are removed. These tend to be residual boilerplate that survived stop-word removal but still lacks discriminatory power.

Adjust the thresholds below and observe the effect on vocabulary size.

In [ ]:
# ═══════════════════════════════════════════════════════
# YOUR CHOICES - adjust and re-run this cell
MIN_DF = 2     # remove tokens appearing in fewer than this many emails
MAX_DF = 0.90  # remove tokens appearing in more than this fraction of emails
# ═══════════════════════════════════════════════════════

# Build per-document token sets
doc_tokens = [
    set(run_pipeline([body], ACTIVE_SW, NUMBER_CHOICE, NORMALISATION_CHOICE))
    for body in corpus["body"]
]
n_docs = len(doc_tokens)

df_counts = Counter()
for doc in doc_tokens:
    for token in doc:
        df_counts[token] += 1

full_vocab = set(t for doc in doc_tokens for t in doc)
after_min  = {t for t in full_vocab if df_counts[t] >= MIN_DF}
after_both = {t for t in after_min  if df_counts[t] / n_docs <= MAX_DF}

print(f"Vocabulary sizes with min_df={MIN_DF}, max_df={MAX_DF:.0%}:")
print(f"  Full (no pruning):     {len(full_vocab):>6,} tokens")
print(f"  After min_df:          {len(after_min):>6,} tokens  ({len(full_vocab)-len(after_min):,} rare terms removed)")
print(f"  After min_df + max_df: {len(after_both):>6,} tokens  ({len(after_min)-len(after_both):,} boilerplate removed)")
print()

# Show what max_df removes - near-universal boilerplate
boilerplate = sorted([(t, df_counts[t]/n_docs)
                       for t in full_vocab if df_counts[t]/n_docs > MAX_DF],
                      key=lambda x: -x[1])[:15]
if boilerplate:
    print(f"Tokens present in more than {MAX_DF:.0%} of emails:")
    for t, pct in boilerplate:
        print(f"  {t:<20} {pct:.0%} of emails")
else:
    print(f"No tokens exceed the {MAX_DF:.0%} threshold at current settings.")

---
## Step 9: Dictionary Comparison

Loughran and McDonald (2011) show that between 73% and 80% of the words flagged as negative by the Harvard General Inquirer carry no negative meaning in a financial context. The cell below applies both dictionaries to the same pipeline output, making this false-positive problem visible in practice.

In [ ]:
texts        = corpus["body"].tolist()
your_tokens  = run_pipeline(texts, ACTIVE_SW, NUMBER_CHOICE, NORMALISATION_CHOICE)
token_set    = set(your_tokens)

# Tokens flagged differently by each dictionary
hv_neg_not_lm = sorted(token_set & HV_NEGATIVE - LM_NEGATIVE)
lm_neg_not_hv = sorted(token_set & LM_NEGATIVE - HV_NEGATIVE)
hv_pos_not_lm = sorted(token_set & HV_POSITIVE - LM_POSITIVE)

print(f"Tokens flagged NEGATIVE by Harvard GI but NOT by LM ({len(hv_neg_not_lm)} total):")
print(f"  {hv_neg_not_lm[:20]}")
print()
print(f"Tokens flagged NEGATIVE by LM but NOT by Harvard GI ({len(lm_neg_not_hv)} total):")
print(f"  {lm_neg_not_hv[:20]}")
print()
print("Many terms in the first list are ordinary business vocabulary that carries")
print("no negative connotation in a financial context.")
print("The second list contains terms that are specifically negative in corporate disclosure.")
print()

# Side-by-side match counts
lm_pos, lm_neg = count_lm_matches(your_tokens)
hv_pos = sum(1 for t in your_tokens if t in HV_POSITIVE)
hv_neg = sum(1 for t in your_tokens if t in HV_NEGATIVE)

fig, ax = plt.subplots(figsize=(9, 4))
fig.patch.set_facecolor("#0f0f1a")
x = np.arange(2)
w = 0.3
ax.bar(x - w/2, [lm_pos, lm_neg], w, color=["#22c55e","#ef4444"], alpha=0.85,
       label="Loughran-McDonald")
ax.bar(x + w/2, [hv_pos, hv_neg], w, color=["#86efac","#fca5a5"], alpha=0.85,
       label="Harvard General Inquirer")
ax.set_xticks(x)
ax.set_xticklabels(["Positive matches", "Negative matches"])
ax.set_title("Dictionary comparison - same pipeline, different lexicons",
             color="#ddd", pad=10)
ax.legend()
ax.set_ylabel("Total matches in corpus")
plt.tight_layout()
plt.show()

if hv_neg > lm_neg:
    ratio = hv_neg / lm_neg
    print(f"Harvard GI finds {ratio:.1f}x more negative matches than LM on this corpus.")
    print("Many of these are likely false positives in a financial text context.")

---
## Step 10: Bias in This Corpus

Section 6.5 of the lecture introduced six forms of corpus bias, drawing on Grimmer, Roberts and Stewart (2022). Consider how each applies to the Enron corpus, and use the chart below alongside Worksheet Questions 8 and 9 to record your analysis.

| Bias type | Definition |
|---|---|
| Resource | Texts over-represent those with the resources to produce and archive them |
| Incentive | Documents are shaped by the strategic incentives of their authors |
| Medium | Platform constraints produce language that differs systematically by source |
| Retrieval | The corpus reflects the retrieval logic as much as the underlying population |
| Survivorship | Only entities that persisted through the sample period are included |
| Self-reporting | Authors chose what to record and how to present it |

In [ ]:
# Examine whether the LM signal reflects financial reality or impression management
period_cats = []
for (period, cat), grp in corpus.groupby(["period","category"]):
    tokens = run_pipeline(grp["body"].tolist(), ACTIVE_SW, NUMBER_CHOICE, NORMALISATION_CHOICE)
    pos, neg = count_lm_matches(tokens)
    total = len(tokens)
    if total > 0:
        period_cats.append({
            "period": period, "category": cat,
            "net": (pos - neg) / total * 100,
            "n_emails": len(grp),
        })

df_pc = pd.DataFrame(period_cats)
pivot = df_pc.pivot(index="category", columns="period", values="net")[
    [c for c in ["pre_crisis","crisis","collapse"] if c in df_pc.columns]
]

fig, ax = plt.subplots(figsize=(11, 5))
fig.patch.set_facecolor("#0f0f1a")
x = np.arange(len(pivot))
period_colors = {"pre_crisis": "#3b82f6", "crisis": "#f59e0b", "collapse": "#ef4444"}
period_labels = {"pre_crisis": "Pre-crisis (2000)", "crisis": "Crisis (2001)",
                 "collapse": "Collapse (late 2001–2)"}
width = 0.25
for i, col in enumerate(pivot.columns):
    offset = (i - len(pivot.columns)/2 + 0.5) * width
    ax.bar(x + offset, pivot[col], width,
           color=period_colors.get(col, "#8b5cf6"),
           label=period_labels.get(col, col), alpha=0.85)
ax.axhline(0, color="#666", linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(pivot.index, rotation=15, ha="right")
ax.set_ylabel("Net LM sentiment (% of tokens)")
ax.set_title("Net LM sentiment by category and period",
             color="#ddd", pad=10)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

print("Executives had the strongest incentive to manage their language strategically.")
print("Does the executive category show the pattern you would expect under incentive bias?")
print("Compare it with the all_staff category, where employees had less control over framing.")

---
## Summary: Your Pipeline Choices

Double-click this cell to fill in your choices and justifications.

| Decision | Options | Your choice | Justification |
|---|---|---|---|
| Tokenisation | whitespace / NLTK | NLTK word_tokenize | |
| Case folding | yes / no | yes | |
| Punctuation removal | yes / no | yes | |
| Stop-word list | none / standard / finance | | |
| Number handling | remove / retain / replace | | |
| Normalisation | none / stemming / lemmatisation | | |
| Min document frequency | integer | | |
| Max document frequency | 0 to 1 | | |

---

**References**

Grimmer, J., Roberts, M.E. and Stewart, B.M. (2022) *Text as Data*. Princeton University Press.

Klimt, B. and Yang, Y. (2004) 'The Enron Corpus', *Machine Learning: ECML 2004*, pp. 217-226.

Loughran, T. and McDonald, B. (2011) 'When is a Liability not a Liability? Textual Analysis, Dictionaries, and 10-Ks', *Journal of Finance*, 66(1), pp. 35-65.

Porter, M.F. (1980) 'An algorithm for suffix stripping', *Program*, 14(3), pp. 130-137.

Renault, T. (2020) 'Sentiment analysis and machine learning in finance: a comparison of methods and models on one million messages', *Digital Finance*, 2(1), pp. 1-13.

Todd, A., Bowden, J. and Moshfeghi, Y. (2024) 'Text-based sentiment analysis in finance: synthesising the existing literature and exploring future directions', *Intelligent Systems in Accounting, Finance and Management*, 31.